# M05-01 — Ranking por ventana

[← Anterior](01-teoria.ipynb) · [Siguiente →](03-lab-acumulados.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

La misma ventana de la teoría, a tamaño NovaShop: top 10 clientes por GMV y, **dentro de cada cliente**, sus 3 productos que más dinero dejan.

No hace falta memorizar la API. En cada paso: ejecuta → mira `rn` → **cambia un número o quita un `partitionBy`** y vuelve a ejecutar. Si solo pegas, no has visto la ventana.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M05-01-ranking-ventana.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Top 10 de la compañía

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Parte de `customer_gmv` (M04-03: una fila por cliente con venta cobrable). Sin `partitionBy`, el ranking es **de toda la empresa**: un solo `rn=1`.

`row_number` pone 1 al GMV más alto (`orderBy desc`), 2 al siguiente, etc. El `where rn <= 10` es el top.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 10 filas, `rn` de 1 a 10, GMV hacia abajo. El nº 1 ronda **6000 €**.

**Por qué este paso.** Si no tienes `customer_gmv`, no es M05: vuelve a M04-03 (groupBy cliente).

**Si no sale.** PATH not found → el Parquet vive en `data/staging/customer_gmv` (lo escribes en el lab de segmentación).


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window

spark = get_spark("novashop-m05")
# Una fila = un cliente (sale de M04-03). Si PATH falla: rehaz ese lab o run_pipeline no basta
# (customer_gmv lo escribes tú en M04-03).
cust = spark.read.parquet(str(STAGING / "customer_gmv"))
print("clientes con GMV cobrable", cust.count())

# Sin partitionBy: un único ranking para toda la tabla
w_global = Window.orderBy(col("gmv").desc())
top10 = (
    cust.withColumn("rn", row_number().over(w_global))  # 1 = el que más factura
    .where(col("rn") <= 10)
)
top10.orderBy("rn").show()


## Prueba tú — Cambia el corte del top

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

En la celda de arriba, cambia `<= 10` por `<= 3` y vuelve a ejecutar. Luego prueba `<= 1`.

**Qué tienes que ver.** `<= 3` da 3 filas. El nº 1 **no cambia** (solo recortas). Si cambia, reordenaste mal.


In [ ]:
print("filas top3", top10.where(col("rn") <= 3).count())  # 3
# ¿El customer_id del rn=1 sigue siendo el mismo que con top 10?
top10.where(col("rn") == 1).select("customer_id", "gmv").show()


### Paso 2 — Top 3 productos **por** cliente

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Ahora el ranking se **reinicia** en cada persona. Eso es `partitionBy("customer_id")`.

Antes hay que **juntar líneas del mismo producto**: si rankeas el fact a palo seco, la misma SKU sale muchas veces (una por línea). Por eso `groupBy(customer_id, product_id)` y luego la ventana.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Como mucho 3 filas por cliente; `rn` 1–3. `filas top3` ≤ 211 × 3. El nº 1 de *ese* cliente es un producto, no el ranking global.

**Por qué este paso.** Si ves 30 filas del mismo cliente, rankeaste líneas: faltó el groupBy producto.


In [ ]:
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))

# Dinero cobrable de cada par cliente+producto (ya no es grano línea)
product_gmv = (
    fact.join(customers, "customer_id", "inner")
    .where(col("is_billable"))
    .groupBy("customer_id", "product_id")
    .agg(fsum("gmv_line").alias("gmv"))
)

# El rn vuelve a 1 en CADA customer_id
w_prod = Window.partitionBy("customer_id").orderBy(col("gmv").desc())
top3 = (
    product_gmv.withColumn("rn", row_number().over(w_prod))
    .where(col("rn") <= 3)
)

# Mira solo al cliente que era nº 1 de la compañía
top_id = top10.select("customer_id").first()["customer_id"]
print("cliente nº 1 de la compañía:", top_id)
top3.where(col("customer_id") == top_id).orderBy("rn").show()
print("filas top3 (todos los clientes)", top3.count())


## Prueba tú — Quita el partitionBy del top 3

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Crea `w_mal = Window.orderBy(col("gmv").desc())` (sin partitionBy), calcula `rn` y filtra `rn <= 3`. Compáralo con `top3`.

**Qué tienes que ver.** Sin `partitionBy`: **3 filas en toda la tabla**. Con él: cientos (3 por cliente). Anota los dos counts en Markdown.


In [ ]:
w_mal = Window.orderBy(col("gmv").desc())  # ranking de TODA la empresa
mal = product_gmv.withColumn("rn", row_number().over(w_mal)).where(col("rn") <= 3)
print("sin partitionBy, filas", mal.count())  # 3 en total, no 3 por cliente
mal.show()
print("con partitionBy, filas", top3.count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Elige un `customer_id` con varios productos y mira sus `rn`.
Empiezan en **1** (no continúan el 1–10 de la compañía). Markdown: id + tres filas.

También: count sin `partitionBy` vs con él (prueba de arriba).


## Mejora — rank vs row_number

Sobre `cust`, añade columnas `row_number`, `rank` y `dense_rank` con el mismo `w_global`. Si hay empate de GMV se ve el salto. Markdown: qué salta y qué no.

Si te atasca, el código está en la celda siguiente.


In [ ]:
from pyspark.sql.functions import rank, dense_rank

cmp_ = (
    cust.withColumn("rn", row_number().over(w_global))
    .withColumn("rk", rank().over(w_global))
    .withColumn("dr", dense_rank().over(w_global))
    .orderBy(col("gmv").desc())
)
cmp_.select("customer_id", "gmv", "rn", "rk", "dr").show(15)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Un solo rn=1 en todo el fact | Olvidaste partitionBy | Añádelo para “por cliente” |
| Top 3 con 30 filas del mismo cliente | Rankeaste líneas | groupBy cliente+producto antes |
| Window sin orderBy | Ranking indefinido | Siempre ordena la métrica |
| No está customer_gmv | Saltaste M04-03 | Ese lab escribe el Parquet |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M05-02 acumulados](03-lab-acumulados.ipynb).
